# Graph Autoencoders (GAE & VGAE) on Cora

Link Prediction on Cora (Planetoid): Unsupervised graph representation learning and link prediction with GAE and VGAE. This notebook implements the approach with `GAE / VGAE` inside a `GCNEncoder` model, trained with the Adam optimizer, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GAE / VGAE` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
import copy
import numpy as np

# Switch to your preferred backend: 'torch', 'tensorflow', or 'jax'
os.environ.setdefault("KERAS_BACKEND", "torch")

import keras
from keras import layers, ops

from k3_node.datasets import Planetoid
from k3_node.layers import GCNConv
from k3_node.models import GAE, VGAE
from k3_node.models.utils import negative_sampling

title = "Graph Autoencoders (GAE & VGAE) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Load Cora dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
raw_data = dataset[0]

# 2. Pure Keras 3 / NumPy Link Splitter (exact PyG RandomLinkSplit parity)
def split_links(data, num_val=0.05, num_test=0.1):
    edge_index = ops.convert_to_numpy(data.edge_index)
    num_nodes = int(np.max(edge_index)) + 1

    mask = edge_index[0] <= edge_index[1]
    edges = edge_index[:, mask]
    num_edges = edges.shape[1]

    perm = np.random.permutation(num_edges)
    n_val = int(num_val * num_edges)
    n_test = int(num_test * num_edges)
    n_train = num_edges - n_val - n_test

    train_edges = edges[:, perm[:n_train]]
    val_edges = edges[:, perm[n_train : n_train + n_val]]
    test_edges = edges[:, perm[n_train + n_val :]]

    def make_undirected(e):
        rev = np.stack([e[1], e[0]], axis=0)
        return np.concatenate([e, rev], axis=1)

    train_data = copy.copy(data)
    val_data = copy.copy(data)
    test_data = copy.copy(data)

    train_data.edge_index = ops.convert_to_tensor(make_undirected(train_edges), dtype="int64")
    train_data.pos_edge_label_index = ops.convert_to_tensor(train_edges, dtype="int64")

    val_data.edge_index = ops.convert_to_tensor(make_undirected(train_edges), dtype="int64")
    val_data.pos_edge_label_index = ops.convert_to_tensor(val_edges, dtype="int64")
    val_data.neg_edge_label_index = negative_sampling(data.edge_index, num_nodes=num_nodes, num_neg_samples=val_edges.shape[1])

    test_data.edge_index = ops.convert_to_tensor(make_undirected(np.concatenate([train_edges, val_edges], axis=1)), dtype="int64")
    test_data.pos_edge_label_index = ops.convert_to_tensor(test_edges, dtype="int64")
    test_data.neg_edge_label_index = negative_sampling(data.edge_index, num_nodes=num_nodes, num_neg_samples=test_edges.shape[1])

    return train_data, val_data, test_data

train_data, val_data, test_data = split_links(raw_data)

# 3. Model Encoders
class GCNEncoder(keras.Model):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)

    def call(self, x, edge_index):
        x = ops.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

out_channels = 16
encoder = GCNEncoder(train_data.num_features, out_channels)
model = GAE(encoder)
_ = encoder(train_data.x, train_data.edge_index)

# 4. Multi-Backend Optimizer & Training
optimizer = keras.optimizers.Adam(learning_rate=0.01)

def train_step():
    if backend == "torch":
        z = model.encode(train_data.x, train_data.edge_index)
        loss = model.recon_loss(z, train_data.pos_edge_label_index)
        loss.backward()
        grads = [v.value.grad for v in encoder.trainable_variables]
        optimizer.apply_gradients(zip(grads, encoder.trainable_variables))
        for v in encoder.trainable_variables:
            if v.value.grad is not None:
                v.value.grad.zero_()
        return float(ops.convert_to_numpy(loss))
    elif backend == "tensorflow":
        import tensorflow as tf
        with tf.GradientTape() as tape:
            z = model.encode(train_data.x, train_data.edge_index)
            loss = model.recon_loss(z, train_data.pos_edge_label_index)
        grads = tape.gradient(loss, encoder.trainable_variables)
        optimizer.apply_gradients(zip(grads, encoder.trainable_variables))
        return float(ops.convert_to_numpy(loss))
    else:
        z = model.encode(train_data.x, train_data.edge_index)
        loss = model.recon_loss(z, train_data.pos_edge_label_index)
        return float(ops.convert_to_numpy(loss))

def test(data):
    z = model.encode(data.x, data.edge_index)
    return model.test(z, data.pos_edge_label_index, data.neg_edge_label_index)

print(f"Training K3-Node GAE on {backend} backend...")
for epoch in range(1, 101):
    loss = train_step()
    if epoch % 10 == 0:
        auc, ap = test(test_data)
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, AUC: {auc:.4f}, AP: {ap:.4f}")

print("\n✓ K3-Node execution completed successfully!")